# AG-mini: Corollary 1 bound against measured exploitability (Figure 1)
AG-mini is a reduction of the attack graph with 4 hosts and 2 servers (64 states) and 48 contexts: the wiring of each server to one of the hosts, which server the attacker targets, and $\lambda \in \{0, 0.5\}$. It is small enough that the exact oracle belief $b(h)$ over (state, context) is computable by Bayes filtering.

1. A recurrent encoder with a posterior readout $q_z$ (one softmax over all 3,072 joint (state, context) classes) is trained by cross-entropy on rollouts with uniform-random defender actions, so the training histories cover the history space broadly. Its cross-entropy optimum is the exact posterior.
2. At each of seven checkpoints, the deployed defender plays QMDP on $q_z$, using the exact $Q^\star(s, \theta, a)$. $\epsilon_b$ is the 95th percentile of $\mathrm{TV}(b(h), q_z(h))$ over its on-policy histories.
3. The predicted bound $\epsilon_\phi + \frac{2R_{sp}}{(1-\gamma)^2}\epsilon_b$ is compared with the measured exploitability $\widehat\Delta = \widehat V^\star - \widehat W$, paired over 1,000 shared episodes, where $\widehat V^\star$ is the best evaluated defender (QMDP on the exact belief).

Conventions:
- **Attacker.** In AG-mini the attacker's type fixes its response: it strikes the host wired to its target server, acting uniformly at random with probability $\lambda$. Each context therefore has a single admissible attacker, and the worst case over attackers is that attacker.
- **$\epsilon_\phi = 0$.** The deployed policy and the reference apply the same QMDP rule, so they differ only in the belief they act on.
- **$R_{sp} = 1.1$**, the realized reward's range: protection in $\{0, 1\}$ minus a 0.1 server-cleaning cost.

## Settings

In [ ]:
import os
import numpy as np
from itertools import permutations
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

GAMMA = 0.9                      # discount for Q*, returns, and the bound
T_HORIZON = 20                   # episode length (train and eval)
TRAIN_EPISODES = 25000           # training iterations (batches)
BATCH_SIZE = 64
CHECKPOINTS = [100, 250, 500, 1000, 5000, 10000, 25000]
EVAL_EPISODES = 1000
EPS_B_PERCENTILE = 95            # eps_b is this percentile of TV(b, q_z) over on-policy histories
EPS_PHI = 0.0                    # the deployed and reference policies share the same QMDP rule
HIDDEN_SIZE = 128
SKIP_TRAINING = False            # True: evaluate existing checkpoints in CHECKPOINT_DIR
CHECKPOINT_DIR = "bound_ckpts"
EVAL_SEED = 12345
TRAIN_SEED = 0

torch.manual_seed(TRAIN_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## AG-mini environment, exact Bayes filter, rewards, and the bound coefficient

In [ ]:
d = 6
n = 2 ** d
U = [[1], [2], [3], [4]]
n_u = 4
A = [[1], [2], [3], [4], [5], [6]]
n_a = 6
TERMINALS = [1, 2, 3, 4]
alpha0 = 0.1
alpha = np.full(d, alpha0)
miss_detec = 0.1
false_detec = 0.1
Len_Theta = 11
LAM_VALS = [0.0, 0.5]

def all_states_bits(d):
    return np.array([[(i >> b) & 1 for b in range(d)] for i in range(2 ** d)])

State_Matrix = all_states_bits(d)

def enumerate_contexts():
    contexts, thetas = [], []
    wirings = list(permutations(TERMINALS, 2))
    for s1_parent, s2_parent in wirings:
        for target_idx in (0, 1):
            for lam in LAM_VALS:
                Theta = np.zeros(Len_Theta)
                Theta[s1_parent - 1] = 1.0
                Theta[4 + (s2_parent - 1)] = 1.0
                Theta[8 + target_idx] = 1.0
                Theta[10] = lam
                contexts.append(dict(s1_parent=s1_parent, s2_parent=s2_parent,
                                     target_idx=target_idx, lam=lam))
                thetas.append(Theta)
    return contexts, np.array(thetas)

contexts, Thetas = enumerate_contexts()
n_theta = len(contexts)
assert n_theta == 48

def Def_Mask(a):
    am = np.zeros(d, dtype=bool)
    for idx in A[a]:
        am[idx - 1] = True
    return am

def Att_Vec(u):
    um = np.zeros(n_u, dtype=bool)
    for idx in U[u]:
        um[idx - 1] = True
    return um

def compromise_prob_next(x, a, u, Theta, alpha):
    x = np.asarray(x, dtype=int)
    a = np.asarray(a, dtype=bool)
    u = np.asarray(u, dtype=bool)
    p = np.zeros(6)
    s1_parent = np.argmax(Theta[0:4])
    s2_parent = np.argmax(Theta[4:8])
    for j in range(4):
        attacked = u[j]
        p[j] = (alpha[j] if a[j] else 1.0) if attacked else (alpha[j] * x[j] if a[j] else float(x[j]))
    for k, parent in enumerate([s1_parent, s2_parent]):
        j = 4 + k
        infected = (x[parent] == 1)
        p[j] = (alpha[j] if a[j] else 1.0) if infected else (alpha[j] * x[j] if a[j] else float(x[j]))
    return p

def Observation(x, rng):
    y = np.zeros(d)
    def emit(compromised):
        if compromised:
            return -1 if rng.random() < miss_detec else 1
        return 1 if rng.random() < false_detec else -1
    for i in range(d):
        y[i] = emit(x[i] == 1)
    return y

def obs_likelihood(y, x_bits_all):
    lik = np.ones(x_bits_all.shape[0])
    for i in range(d):
        compromised = x_bits_all[:, i] == 1
        p_y = np.where(compromised, 1 - miss_detec, false_detec) if y[i] == 1 \
              else np.where(compromised, miss_detec, 1 - false_detec)
        lik *= p_y
    return lik

def transition_matrix(a_idx, u_idx, Theta):
    a_mask, u_mask = Def_Mask(a_idx), Att_Vec(u_idx)
    T = np.zeros((n, n))
    for s in range(n):
        eta = compromise_prob_next(State_Matrix[s, :], a_mask, u_mask, Theta, alpha)
        node_probs = np.where(State_Matrix, eta, 1.0 - eta)
        T[s, :] = node_probs.prod(axis=1)
    return T

def attacker_policy_probs(Theta):
    lam = Theta[10]
    target_idx = int(np.argmax(Theta[8:10]))
    s1_parent = int(np.argmax(Theta[0:4]))
    s2_parent = int(np.argmax(Theta[4:8]))
    parent = s1_parent if target_idx == 0 else s2_parent
    br = np.zeros(n_u)
    br[parent] = 1.0
    uniform = np.full(n_u, 1.0 / n_u)
    return lam * uniform + (1 - lam) * br

print("Precomputing transition matrices for all (theta, a, u)...")
Trans = {}
for ti, Theta in enumerate(Thetas):
    for a_idx in range(n_a):
        for u_idx in range(n_u):
            Trans[(ti, a_idx, u_idx)] = transition_matrix(a_idx, u_idx, Theta)
print(f"Done: {len(Trans)} matrices of shape {n}x{n}.")

print("Precomputing action-marginalized transition matrices per (theta, a)...")
Marg = {}
for ti, Theta in enumerate(Thetas):
    probs_u = attacker_policy_probs(Theta)
    for a_idx in range(n_a):
        T_sum = np.zeros((n, n))
        for u_idx in range(n_u):
            if probs_u[u_idx] > 0:
                T_sum += probs_u[u_idx] * Trans[(ti, a_idx, u_idx)]
        Marg[(ti, a_idx)] = T_sum
print(f"Done: {len(Marg)} marginalized matrices.")

def sanity_checks():
    for key, T in Trans.items():
        assert np.allclose(T.sum(axis=1), 1.0, atol=1e-6), f"Transition matrix {key} rows don't sum to 1."
    for key, T in Marg.items():
        assert np.allclose(T.sum(axis=1), 1.0, atol=1e-6), f"Marginal matrix {key} rows don't sum to 1."
    rng_check = np.random.default_rng(123)
    for _ in range(1000):
        y_test = rng_check.choice([-1, 1], size=d)
        lik = obs_likelihood(y_test, State_Matrix)
        assert np.all(lik >= 0) and np.all(lik <= 1), "Observation likelihood out of [0,1]."
    print("Sanity checks passed: transition and marginal rows sum to 1, likelihoods in [0,1].")

def exact_belief_update(b, a_idx, y):
    b_next = np.zeros_like(b)
    lik = obs_likelihood(y, State_Matrix)
    for ti in range(n_theta):
        T = Marg[(ti, a_idx)]
        b_pred = b[:, ti] @ T
        b_next[:, ti] = b_pred * lik
    total = b_next.sum()
    assert total > 0, "Degenerate update: zero-probability observation under the model."
    return b_next / total



def action_cost(a_idx):
    return 0.1 if (a_idx == 4 or a_idx == 5) else 0.0

def target_server_node(ti):
    return 4 + contexts[ti]['target_idx']  

print("Building expected stage-reward table r[ti, s, a]...")
r_table = np.zeros((n_theta, n, n_a))
for ti in range(n_theta):
    tgt = target_server_node(ti)
    protection_next = 1.0 - State_Matrix[:, tgt]       
    for a_idx in range(n_a):
        exp_prot = Marg[(ti, a_idx)] @ protection_next  
        r_table[ti, :, a_idx] = exp_prot - action_cost(a_idx)
R_MAX = r_table.max()
R_MIN = r_table.min()
RSP = 1.0 - (-0.1)   
print(f"Expected-reward table range [{R_MIN:.4f}, {R_MAX:.4f}]; using Rsp = {RSP:.2f} "
      f"(analytic realized-reward range).")

BOUND_COEF = 2.0 * RSP / (1.0 - GAMMA) ** 2
print(f"Corollary 1 coefficient 2*Rsp/(1-gamma)^2 = {BOUND_COEF:.1f} at gamma = {GAMMA}.")

## Exact $Q^\star$ by value iteration, and the QMDP action rule

In [ ]:
def solve_q_star(tol=1e-10, max_iter=10000):
    """Per-context value iteration; the defender MAXIMIZES its reward."""
    Q = np.zeros((n_theta, n, n_a))
    for ti in range(n_theta):
        V = np.zeros(n)
        T_a = np.stack([Marg[(ti, a_idx)] for a_idx in range(n_a)])  
        r_a = r_table[ti].T                                            
        for _ in range(max_iter):
            Q_ti = r_a + GAMMA * (T_a @ V)                          
            V_new = Q_ti.max(axis=0)
            if np.abs(V_new - V).max() < tol:
                V = V_new
                break
            V = V_new
        Q[ti] = (r_a + GAMMA * (T_a @ V)).T                           
    return Q

print("Solving Q*(s, theta, a) by exact value iteration...")
Q_STAR = solve_q_star()
print(f"Done. V* range over (s, theta): "
      f"[{Q_STAR.max(axis=2).min():.3f}, {Q_STAR.max(axis=2).max():.3f}].")

Q_STAR_SNT = np.transpose(Q_STAR, (1, 0, 2)).copy()   # (n, n_theta, n_a)

def qmdp_action(belief_joint):
    """belief_joint: (n, n_theta). Returns argmax_a E_belief[Q*(s,theta,a)]."""
    scores = np.einsum('st,sta->a', belief_joint, Q_STAR_SNT)
    return int(np.argmax(scores))

## Recurrent encoder with a joint posterior readout

In [ ]:
INPUT_SIZE = n_a + d
N_JOINT = n * n_theta          # 3072

def joint_class(s_idx, ti):
    return s_idx * n_theta + ti

class RecurrentEncoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)

    def forward(self, seq, h0=None):
        out, h_n = self.gru(seq, h0)
        return out, h_n

class JointHead(nn.Module):
    """Single softmax over all 3072 joint (s, theta) classes; CE optimum is
    the exact posterior b(h) (proper scoring rule)."""
    def __init__(self, hidden_size, n_joint):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.ReLU(),
            nn.Linear(256, n_joint),
        )

    def forward(self, z):
        return self.net(z)          # logits (…, N_JOINT)

encoder = RecurrentEncoder(INPUT_SIZE, HIDDEN_SIZE).to(device)
joint_head = JointHead(HIDDEN_SIZE, N_JOINT).to(device)
optimizer = optim.Adam(list(encoder.parameters()) + list(joint_head.parameters()), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=8000, gamma=0.5)
ce_loss = nn.CrossEntropyLoss()

## Training: rollouts with uniform-random defender actions, so the histories cover the space broadly

In [ ]:
def sample_episode(T_horizon, rng):
    ti_true = rng.integers(n_theta)
    Theta_true = Thetas[ti_true]
    probs_u_true = attacker_policy_probs(Theta_true)
    s_true = rng.integers(n)

    inputs = np.zeros((T_horizon, INPUT_SIZE), dtype=np.float32)
    joint_labels = np.zeros(T_horizon, dtype=np.int64)

    for t in range(T_horizon):
        a_idx = rng.integers(n_a)
        u_idx = rng.choice(n_u, p=probs_u_true)

        T = Trans[(ti_true, a_idx, u_idx)]
        s_next = rng.choice(n, p=T[s_true, :])
        y = Observation(State_Matrix[s_next, :], rng)

        inputs[t, a_idx] = 1.0
        inputs[t, n_a:] = y
        joint_labels[t] = joint_class(s_next, ti_true)
        s_true = s_next

    return inputs, joint_labels

def train():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    rng = np.random.default_rng(TRAIN_SEED)
    ckpt_set = set(CHECKPOINTS)
    losses = []

    for ep in range(TRAIN_EPISODES):
        batch_inputs, batch_labels = [], []
        for _ in range(BATCH_SIZE):
            inputs, joint_labels = sample_episode(T_HORIZON, rng)
            batch_inputs.append(inputs)
            batch_labels.append(joint_labels)

        seq = torch.tensor(np.stack(batch_inputs), device=device)              # (B, T, in)
        targets = torch.tensor(np.stack(batch_labels), device=device)          # (B, T)
        out, _ = encoder(seq)                                                  # (B, T, H)
        logits = joint_head(out)                                               # (B, T, N_JOINT)
        loss = ce_loss(logits.reshape(-1, N_JOINT), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

        if (ep + 1) % 1000 == 0:
            print(f"episode {ep+1}: loss {np.mean(losses[-1000:]):.4f}")
        if (ep + 1) in ckpt_set:
            torch.save(dict(encoder=encoder.state_dict(),
                            joint_head=joint_head.state_dict()),
                       f"{CHECKPOINT_DIR}/ckpt_{ep+1}.pt")
            print(f"episode {ep+1}: checkpoint saved")
    return losses

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    encoder.load_state_dict(ckpt["encoder"])
    joint_head.load_state_dict(ckpt["joint_head"])
    encoder.eval()
    joint_head.eval()

## Posterior readout and diagnostics (TV, entropy, KL)

In [ ]:
class IncrementalPosterior:
    """Carries the GRU hidden state forward; one forward pass per step."""
    def __init__(self):
        self.h = None

    def step(self, a_idx, y):
        x = np.zeros((1, 1, INPUT_SIZE), dtype=np.float32)
        x[0, 0, a_idx] = 1.0
        x[0, 0, n_a:] = y
        with torch.no_grad():
            out, self.h = encoder(torch.tensor(x, device=device), self.h)
            logits = joint_head(out[:, -1, :])
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        return probs.reshape(n, n_theta)      # joint class = s * n_theta + ti

def qz_nested(q_joint):
    """Nested form of Proposition 3, recovered exactly from the joint."""
    q_theta = q_joint.sum(axis=0)
    q_s_given_theta = np.where(q_theta > 0, q_joint / np.maximum(q_theta, 1e-300), 1.0 / n)
    return q_theta, q_s_given_theta

def total_variation(b, q):
    return 0.5 * np.abs(b - q).sum()

def entropy_nats(p):
    """H(p) in nats; 0*log0 = 0."""
    p = p.ravel()
    nz = p > 0
    return float(-(p[nz] * np.log(p[nz])).sum())

def kl_nats(b, q, q_floor=1e-30):
    """KL(b || q) in nats; b-zeros contribute 0, q floored for numerical stability."""
    b = b.ravel()
    q = np.maximum(q.ravel(), q_floor)
    nz = b > 0
    return float((b[nz] * (np.log(b[nz]) - np.log(q[nz]))).sum())

## Shared evaluation episodes and rollouts

In [ ]:
def draw_eval_episodes(n_episodes, seed):
    rng = np.random.default_rng(seed)
    specs = []
    for i in range(n_episodes):
        ti = int(rng.integers(n_theta))
        s0 = int(rng.integers(n))
        probs_u = attacker_policy_probs(Thetas[ti])
        u_seq = rng.choice(n_u, size=T_HORIZON, p=probs_u)
        env_seed = int(rng.integers(2 ** 31 - 1))
        specs.append(dict(ti=ti, s0=s0, u_seq=u_seq, env_seed=env_seed))
    return specs

def rollout(spec, policy, track_diag=False):
    """policy: 'exact' (QMDP on exact belief) or 'learned' (QMDP on q_z).
    Returns (G, tvs, hbs, kls): discounted return and, if track_diag,
    per-step TV(b, q), H(b), KL(b || q)."""
    ti, s = spec['ti'], spec['s0']
    rng = np.random.default_rng(spec['env_seed'])
    tgt = target_server_node(ti)

    b = np.full((n, n_theta), 1.0 / (n * n_theta))
    inc = IncrementalPosterior() if (policy == 'learned' or track_diag) else None
    q = np.full((n, n_theta), 1.0 / (n * n_theta))

    G, tvs, hbs, kls = 0.0, [], [], []
    for t in range(T_HORIZON):
        acting_belief = b if policy == 'exact' else q
        a_idx = qmdp_action(acting_belief)
        u_idx = int(spec['u_seq'][t])

        T = Trans[(ti, a_idx, u_idx)]
        s_next = int(rng.choice(n, p=T[s, :]))
        y = Observation(State_Matrix[s_next, :], rng)

        r = (1.0 - State_Matrix[s_next, tgt]) - action_cost(a_idx)
        G += (GAMMA ** t) * r

        b = exact_belief_update(b, a_idx, y)
        if inc is not None:
            q = inc.step(a_idx, y)
            if track_diag:
                tvs.append(total_variation(b, q))
                hbs.append(entropy_nats(b))
                kls.append(kl_nats(b, q))
        s = s_next

    return G, tvs, hbs, kls

def evaluate_checkpoint(specs):
    """Deployed (learned-posterior QMDP) policy on shared episodes.
    Returns per-episode returns, eps_b, and diagnostic arrays."""
    returns, tv_all, hb_all, kl_all = [], [], [], []
    for spec in specs:
        G, tvs, hbs, kls = rollout(spec, policy='learned', track_diag=True)
        returns.append(G)
        tv_all.extend(tvs)
        hb_all.extend(hbs)
        kl_all.extend(kls)
    returns = np.array(returns)
    tv_all, hb_all, kl_all = map(np.array, (tv_all, hb_all, kl_all))
    eps_b = float(np.percentile(tv_all, EPS_B_PERCENTILE))
    return returns, eps_b, tv_all, hb_all, kl_all

def evaluate_exact_reference(specs):
    returns = []
    for spec in specs:
        G, _, _, _ = rollout(spec, policy='exact', track_diag=False)
        returns.append(G)
    return np.array(returns)

## Spearman rank correlation

In [ ]:
def _ranks(x):
    x = np.asarray(x, dtype=float)
    order = np.argsort(x)
    ranks = np.empty(len(x))
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0   # average rank for ties
        i = j + 1
    return ranks

def spearman(x, y):
    rx, ry = _ranks(x), _ranks(y)
    rx = rx - rx.mean()
    ry = ry - ry.mean()
    denom = np.sqrt((rx ** 2).sum() * (ry ** 2).sum())
    return float((rx * ry).sum() / denom) if denom > 0 else float('nan')

## Train

In [ ]:
sanity_checks()
if SKIP_TRAINING:
    print(f"SKIP_TRAINING = True: evaluating existing checkpoints in {CHECKPOINT_DIR} "
          "(HIDDEN_SIZE must match how they were trained).")
else:
    losses = train()

## Evaluate every checkpoint against the exact-belief reference

In [ ]:
specs = draw_eval_episodes(EVAL_EPISODES, EVAL_SEED)

ref_returns = evaluate_exact_reference(specs)
print(f"exact-belief QMDP: mean return {ref_returns.mean():.4f} "
      f"(se {ref_returns.std(ddof=1) / np.sqrt(len(ref_returns)):.4f})")

rows = []
all_returns = {'exact': ref_returns}
for ep in CHECKPOINTS:
    load_checkpoint(f"{CHECKPOINT_DIR}/ckpt_{ep}.pt")
    returns, eps_b, tvs, hbs, kls = evaluate_checkpoint(specs)
    all_returns[ep] = returns
    rows.append(dict(ckpt=ep, eps_b=eps_b, tv_mean=float(tvs.mean()), tv_max=float(tvs.max()),
                     hb_mean=float(hbs.mean()), kl_mean=float(kls.mean()),
                     W=float(returns.mean()), W_se=float(returns.std(ddof=1) / np.sqrt(len(returns)))))
    print(f"ckpt {ep}: mean TV {tvs.mean():.4f} | eps_b (p{EPS_B_PERCENTILE}) {eps_b:.4f} | max TV {tvs.max():.4f} | "
          f"W {returns.mean():.4f} | E[H(b)] {hbs.mean():.4f} | E[KL(b||q)] {kls.mean():.4f}")

# Entropy floor: the lowest cross-entropy any encoder can reach on these histories is E[H(b)];
# the final encoder sits E[KL(b||q)] above it.
last = rows[-1]
print(f"\nEntropy floor (ckpt {last['ckpt']}): E[H(b)] = {last['hb_mean']:.4f} nats, "
      f"excess E[KL(b||q)] = {last['kl_mean']:.4f} nats "
      f"({100.0 * last['kl_mean'] / (last['hb_mean'] + last['kl_mean']):.1f}% of the achieved cross-entropy is removable in principle).")

# V_hat* = best evaluated defender on the shared episodes; Delta_hat is paired per episode
best_key = max(all_returns, key=lambda k: float(all_returns[k].mean()))
V_hat_star = float(all_returns[best_key].mean())
print(f"\nVhat* (best evaluated defender = {best_key}) = {V_hat_star:.4f}")
for r in rows:
    diffs = all_returns[best_key] - all_returns[r['ckpt']]
    r['delta_hat'] = float(diffs.mean())
    r['delta_se'] = float(diffs.std(ddof=1) / np.sqrt(len(diffs)))
    r['bound'] = EPS_PHI + BOUND_COEF * r['eps_b']

rho = spearman([r['bound'] for r in rows], [r['delta_hat'] for r in rows])
below = all(r['delta_hat'] <= r['bound'] for r in rows)

print(f"\n{'ckpt':>7} {'eps_b':>8} {'bound':>10} {'W':>9} {'Delta_hat':>10} {'(pairedSE)':>11} {'E[H(b)]':>8} {'E[KL]':>8}")
for r in rows:
    print(f"{r['ckpt']:>7} {r['eps_b']:>8.4f} {r['bound']:>10.2f} {r['W']:>9.4f} {r['delta_hat']:>10.4f} "
          f"{r['delta_se']:>11.4f} {r['hb_mean']:>8.4f} {r['kl_mean']:>8.4f}")
print(f"\nSpearman rho(bound, Delta_hat) = {rho:.3f}")
print(f"All points below y = x (bound holds empirically): {below}")

## Figure 1

In [ ]:
pred = np.array([r['bound'] for r in rows])
meas = np.maximum(np.array([r['delta_hat'] for r in rows]), 1e-4)     # log axis needs a positive floor
err = 1.96 * np.array([r['delta_se'] for r in rows])
labels = [str(r['ckpt']) for r in rows]

fig, ax = plt.subplots(figsize=(5.6, 4.6))

# main panel: zoomed to the checkpoints, with independent x and y limits
y_lo_data = max(min((meas - err).min(), meas.min()), 1e-3)
x_lo, x_hi = pred.min() / 1.25, pred.max() * 1.25
y_lo, y_hi = y_lo_data / (10 ** 0.35), meas.max() * 2.2
err_lo = np.minimum(err, meas - y_lo * 1.001)                         # keep the bars inside the log frame
ax.errorbar(pred, meas, yerr=[err_lo, err], fmt='o', ms=6, capsize=3, zorder=3, color='tab:blue')
for i, (x_, y_, lb) in enumerate(zip(pred, meas, labels)):
    ax.annotate(lb, (x_, y_), textcoords="offset points", xytext=(7, 6) if i % 2 == 0 else (7, -11), fontsize=9)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(x_lo, x_hi); ax.set_ylim(y_lo, y_hi)
ax.set_xlabel(r'Predicted bound  $\epsilon_\phi + \frac{2R_{sp}}{(1-\gamma)^2}\epsilon_b$')
ax.set_ylabel(r'Measured exploitability  $\widehat{\Delta}$')
ax.grid(True, which='both', alpha=0.15)

# inset: the full square range with the y = x line, so every point is visibly below it
axin = ax.inset_axes([0.60, 0.10, 0.36, 0.38])
lo, hi = min(meas.min(), 1e-2) * 0.5, pred.max() * 3.0
axin.plot([lo, hi], [lo, hi], 'k--', lw=1)
axin.scatter(pred, meas, s=10, color='tab:blue', zorder=3)
axin.set_xscale('log'); axin.set_yscale('log')
axin.set_xlim(lo, hi); axin.set_ylim(lo, hi)
axin.tick_params(labelsize=6)
axin.set_title('full range, y = x', fontsize=7)
axin.grid(True, which='major', alpha=0.15)

fig.tight_layout()
plt.savefig('agmini_bound.jpg', dpi=300, bbox_inches='tight')
plt.show()
print(f"Spearman rho = {rho:.2f}; labels are training episodes; bars are +/-1.96 paired SE")